# Introduction

### About Olist E-commerce Company

Olist is the largest department store in Brazilian marketplaces. Olist connects small businesses from all over Brazil to channels without hassle and with a single contract. Those merchants are able to sell their products through the Olist Store and ship them directly to the customers using Olist logistics partners. The Brazilian real (pl. reais; sign: R$; code: BRL) is the official currency of Brazil.

### Dataset Overview

This is real commercial data, a public dataset of orders made at Olist Store. The dataset has information of 100k orders from 2016 to 2018 made at multiple marketplaces in Brazil. 
Dataset Source: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

After a customer makes a purchase from Olist Pltform, then the seller gets a fulfillment order notification. Once the customer receives the product, or the estimated delivery date is due, the customer gets a satisfaction survey by email where he can give a note for the purchase experience and write down some comments.

**Part 1 - Data Preparation / Cleaning Notebook**: Title:Capstone Project Brazilian E-Commerce Public Cleaning

**Part 2 - Exploratory Data Analysis Notebook**: Title:Capstone Project Brazilian E-Commerce Public EDA, link: http://localhost:8888/notebooks/XDI%20Capstone%20Project%2FCapstone%20Project%20Brazilian%20E-Commerce%20Public%20EDA.ipynb

### Business Problems to Solve

* Operational Efficiency: Finding bottlenecks, wasted spend, or areas where processes can be streamlined.
* Financial Performance: Tracking KPIs and spotting revenue leaks.
* Seller Behavior: Who’s selling, what they’re selling, and where are they struggling. 
* Customer Behavior: Who’s buying, what they're purchasing, and what customer region purchase the most. 

### Data Cleaning Quality Reports

1. Merging Tables
2. Missing Values Handling
3. Duplicate Removal
4. Outlier Treatment
5. Category Standardization
6. Feature Engineering


### Data Preparation / Cleaning

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plot
import seaborn as sns
from pathlib import Path
import re

In [2]:
file_path = Path('/Users/abrahamdunmeyer/Desktop/XDI Data Analyst/Capstone Project/Brazilian E-Commerce Public Dataset by Olist/Dataset/Brazilian E-Commerce Public Dataset by Olist')

data_sets = []

for file in file_path.iterdir():
    fil_string = str(file)
    data_sets.append(fil_string)

Indexing the 9 datasets for a broad look at all datapoints: (table_name):(index number)

1. olist_sellers_dataset: 0
2. product_category_name_translation: 1
3. olist_orders_dataset: 2
4. olist_order_items_dataset: 3
5. olist_customers_dataset: 4
6. olist_geolocation_dataset: 5
7. olist_order_payments_dataset: 6
8. olist_order_reviews_dataset: 7
9. olist_products_dataset: 8

### Import Data

In [3]:
#Building the merchant table to use
olist_sellers_dataset = pd.read_csv(data_sets[0]) #merged on Seller_id
olist_order_items_dataset = pd.read_csv(data_sets[3]) #merged on Seller_id
olist_order_payments_dataset = pd.read_csv(data_sets[6]) #merged on order_id
olist_orders_dataset = pd.read_csv(data_sets[2]) #merged on order_id
olist_customers_dataset = pd.read_csv(data_sets[4])#customer_id

In [4]:
#Product Orders / delivery logistics
olist_products_dataset = pd.read_csv(data_sets[8]) #product ID
product_category_name_translation = pd.read_csv(data_sets[1]) #merg on product name
olist_order_reviews_dataset = pd.read_csv(data_sets[7]) #order_id

### Merge Tables

Consolidating all 9 tables into one big main overview table.

In [5]:
df = olist_sellers_dataset.merge(olist_order_items_dataset, on='seller_id')
df = df.merge(olist_order_payments_dataset, on='order_id')
df = df.merge(olist_orders_dataset, on='order_id')
df = df.merge(olist_customers_dataset, on='customer_id')
df = df.merge(olist_products_dataset, on='product_id')
df = df.merge(product_category_name_translation, on='product_category_name')
df = df.merge(olist_order_reviews_dataset, on='order_id')

### Shape Of My Dataframe

Finding out what are the demensions of the Dataframe, outputting the Dataframe's Rows & Columns.

In [6]:
df.shape

(115609, 40)

### Column Option

To View all columns.

In [26]:
pd.set_option('display.max_columns', None)#if I need to revert back: pd.reset_option('display.max_columns')

### Copy Dataframe

Creating a copy to avoid impacting the raw data.

In [33]:
df_1 = df.copy() #create a copy to avoid impacting the raw data

### Column and Row Overview

See top five rows of our dataset.

In [34]:
df_1.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state,order_id,order_item_id,product_id,shipping_limit_date,price,freight_value,payment_sequential,payment_type,payment_installments,payment_value,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,4a90af3e85dd563884e2afeab1091394,1,ffb64e34a37740dafb6c88f1abd1fa61,2017-08-25 20:50:19,106.20,9.56,1,credit_card,2,115.76,9d6837f9700a3441e7183bff3bc4eef0,delivered,2017-08-21 20:35:44,2017-08-21 20:50:19,2017-08-29 20:33:29,2017-08-30 16:07:13,2017-09-01 00:00:00,f421a2a66b69dbfe6db0c87845281a90,4661,sao paulo,SP,esporte_lazer,26.0,417.0,3.0,700.0,43.0,15.0,35.0,sports_leisure,88980a9c50a6909fa1fe35ddab8fa1e2,5,NaN,NaN,2017-08-31 00:00:00,2017-08-31 21:37:39
1,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,6d953888a914b67350d5bc4d48f2acab,1,f4621f8ad6f54a2e3c408884068be46d,2017-05-11 16:25:11,101.70,15.92,1,credit_card,2,117.62,a973c4e3ad82777add3fa188f91dacea,delivered,2017-05-05 16:12:29,2017-05-05 16:25:11,2017-05-12 05:43:55,2017-06-02 16:57:44,2017-05-30 00:00:00,b4527423469300ee354458e1b5f961be,32223,contagem,MG,esporte_lazer,27.0,485.0,2.0,600.0,35.0,15.0,28.0,sports_leisure,b9b791819c5c1a5c6a4ffc7881f97fb5,1,NaN,"O pedido foi realizado no dia 5/5/2017, e até ...",2017-06-01 00:00:00,2017-06-01 12:57:10
2,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,bc8a5de6abf5b14f98a6135a7fb46731,2,325a06bcce0da45b7f4ecf2797dd40e4,2017-09-05 12:50:19,10.80,2.42,1,credit_card,4,158.80,1554ffe702931a062b4383b109accf63,delivered,2017-08-30 11:47:52,2017-08-30 12:50:19,2017-08-30 19:22:18,2017-09-01 16:51:26,2017-09-20 00:00:00,af0f26435fade1ca984d9affda307199,9310,maua,SP,esporte_lazer,44.0,1089.0,1.0,300.0,16.0,5.0,15.0,sports_leisure,cc77a6d63753c1d7b88b7c64630b97b9,3,NaN,coprei tres ítens faltou entregar um,2017-09-02 00:00:00,2017-09-03 17:31:14
3,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,035201c3c82a97f8a25dd6bd5244b2d5,1,1c36a5285f7f3b1ed2637d7c528ce5ff,2017-11-22 04:30:29,89.99,45.09,1,boleto,1,135.08,9facbfd2dd51a45404d58154b12ed2dd,delivered,2017-11-10 16:54:13,2017-11-14 04:31:07,2017-11-14 20:06:59,2017-11-21 23:26:35,2017-12-04 00:00:00,be1401bbfd64455c798bb4683e915c61,22050,rio de janeiro,RJ,malas_acessorios,21.0,769.0,4.0,1600.0,55.0,37.0,30.0,luggage_accessories,27a894ac7d70600fd49d2ac3b910e65d,5,NaN,A cor é muito diferente da foto.\r\nÉ uma ótim...,2017-11-22 00:00:00,2017-11-23 02:14:56
4,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,0504447548229e075dea8441b37b1e2a,1,8852eb03b04ec3268a66e9b696e25f6f,2017-10-06 02:14:42,199.90,21.89,1,boleto,1,221.79,4e2c1f15de98416a90c2ee06b55ccc9b,delivered,2017-09-28 19:31:57,2017-09-30 02:14:42,2017-10-02 19:33:19,2017-10-06 17:03:59,2017-11-03 00:00:00,9a70db677c8e4e9e25e36729362fe756,97700,santiago,RS,papelaria,30.0,832.0,5.0,2000.0,25.0,55.0,40.0,stationery,265e6d99461ed7db35a7f48a00df7e6c,5,NaN,excelente aquisição,2017-10-07 00:00:00,2017-10-09 10:41:34


### Removing Columns

Cleaning and dropping non-essential columns like 'product_category_name'(names are portuguese, therefore  I want english names).


In [35]:
df_1 = df_1.drop(['product_category_name', 'review_comment_title', 'review_comment_message'], axis=1)
df_1.shape

(115609, 37)

### Assigning Data Types

Ensure non-object items are assigned their proper datatype.

In [12]:
#Fix our dates from an object to an actual date type for later engagements
df_1['shipping_limit_date'] = pd.to_datetime(df_1['shipping_limit_date'])
df_1['order_purchase_timestamp'] = pd.to_datetime(df_1['order_purchase_timestamp'])
df_1['order_approved_at'] = pd.to_datetime(df_1['order_approved_at'])
df_1['order_delivered_carrier_date'] = pd.to_datetime(df_1['order_delivered_carrier_date'])
df_1['order_delivered_customer_date'] = pd.to_datetime(df_1['order_delivered_customer_date'])
df_1['order_estimated_delivery_date'] = pd.to_datetime(df_1['order_estimated_delivery_date'])
df_1['review_creation_date'] = pd.to_datetime(df_1['review_creation_date'])
df_1['review_answer_timestamp'] = pd.to_datetime(df_1['review_answer_timestamp'])


### Feature Engineering (Add Columns)

Breaking up the dates for quicker extraction when performing EDA

**Setting time data types**

In [43]:
df_1['order_year'] =  pd.to_datetime(df_1['order_purchase_timestamp']).dt.year
df_1['order_month'] = pd.to_datetime(df_1['order_purchase_timestamp']).dt.month
df_1['order_day'] = pd.to_datetime(df_1['order_purchase_timestamp']).dt.day
df_1['order_hour'] = pd.to_datetime(df_1['order_purchase_timestamp']).dt.hour
df_1['day_of_week'] = pd.to_datetime(df_1['order_purchase_timestamp']).dt.day_name()
df_1['year_month'] = pd.to_datetime(df_1['order_purchase_timestamp']).dt.to_period('M')
df_1['total_order_value'] = df_1['price'] + df_1['freight_value'] 

**Listing full country name**

In [46]:
state_map = {
    'AC': 'Acre', 'AL': 'Alagoas', 'AM': 'Amazonas', 'AP': 'Amapá', 'BA': 'Bahia',
    'CE': 'Ceará', 'DF': 'Distrito Federal', 'ES': 'Espírito Santo', 'GO': 'Goiás',
    'MA': 'Maranhão', 'MG': 'Minas Gerais', 'MS': 'Mato Grosso do Sul',
    'MT': 'Mato Grosso', 'PA': 'Pará', 'PB': 'Paraíba', 'PE': 'Pernambuco',
    'PI': 'Piauí', 'PR': 'Paraná', 'RJ': 'Rio de Janeiro', 'RN': 'Rio Grande do Norte',
    'RO': 'Rondônia', 'RR': 'Roraima', 'RS': 'Rio Grande do Sul', 'SC': 'Santa Catarina',
    'SE': 'Sergipe', 'SP': 'São Paulo', 'TO': 'Tocantins'
}
df_1['customer_state_name'] = df_1['customer_state'].map(state_map)
df_1['seller_state_name'] = df_1['seller_state'].map(state_map)

**Listing Country Region**

In [49]:
region_map = {
    'AC': 'North', 'AM': 'North', 'AP': 'North', 'PA': 'North', 'RO': 'North', 'RR': 'North', 'TO': 'North',
    'AL': 'Northeast', 'BA': 'Northeast', 'CE': 'Northeast', 'MA': 'Northeast', 'PB': 'Northeast', 
    'PE': 'Northeast', 'PI': 'Northeast', 'RN': 'Northeast', 'SE': 'Northeast',
    'DF': 'Central-West', 'GO': 'Central-West', 'MS': 'Central-West', 'MT': 'Central-West',
    'ES': 'Southeast', 'MG': 'Southeast', 'RJ': 'Southeast', 'SP': 'Southeast',
    'PR': 'South', 'RS': 'South', 'SC': 'South'
}

df_1['customer_region'] = df_1['customer_state'].map(region_map)
df_1['seller_region'] = df_1['seller_state'].map(region_map)

**Note:** 'total_order_value' represents an individual item price, so if you order 5 items, those five items has an independent price.

### Replacing Named Values

Replace payment type's row value name 'boleto' its english translation


In [37]:
df_1['payment_type'] = df_1['payment_type'].replace('boleto','bank_ticket')

### Fixing Missing Value

Values that require an assigned values receives one. There are NaN values in date columns ("order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"), however, adjusting may impact its reasoning for being empty.

In [36]:
#Fixing missing values 
df_1.isnull().sum() #Checked for Null values

df_1['product_height_cm'] = df_1['product_height_cm'].fillna(0)
df_1['product_length_cm'] = df_1['product_length_cm'].fillna(0)
df_1['product_weight_g'] = df_1['product_weight_g'].fillna(0)
df_1['product_width_cm'] = df_1['product_width_cm'].fillna(0)


In [44]:
df_1[df_1[['order_status','order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date']].isnull()]
# count of 14 = df_1[df_1['order_approved_at'].isnull()].count()
deliveries_unknown_dates = df_1[df_1['order_delivered_customer_date'].isnull()]


### Check for Duplicates

Find and remove and duplicated values within the Dataframe.

In [38]:
df_1.duplicated().sum()

np.int64(0)

### Checking False Spelling

Searching columns where row values contain wrong spellling.

In [24]:
category_count = df_1['product_category_name_english'].value_counts()

category_count.to_string()


'product_category_name_english\nbed_bath_table                             11847\nhealth_beauty                               9944\nsports_leisure                              8942\nfurniture_decor                             8743\ncomputers_accessories                       8105\nhousewares                                  7331\nwatches_gifts                               6161\ntelephony                                   4692\ngarden_tools                                4558\nauto                                        4356\ntoys                                        4246\ncool_stuff                                  3964\nperfumery                                   3575\nbaby                                        3178\nelectronics                                 2827\nstationery                                  2607\nfashion_bags_accessories                    2159\npet_shop                                    2020\noffice_furniture                            1773\nconsoles_games    

In [40]:
print(f"Customer State Names\n{df_1['customer_state'].value_counts()}")
print(f"Seller State Names\n{df_1['seller_state'].value_counts()}")

Customer State Names
customer_state
SP    48797
RJ    14987
MG    13429
RS     6413
PR     5879
SC     4218
BA     3942
DF     2449
GO     2359
ES     2300
PE     1851
CE     1527
MT     1106
PA     1081
MS      845
MA      832
PB      619
PI      561
RN      560
AL      455
SE      393
TO      333
RO      279
AM      168
AC       93
AP       83
RR       50
Name: count, dtype: int64
Seller State Names
seller_state
SP    82417
MG     9014
PR     8964
RJ     4906
SC     4221
RS     2224
DF      937
BA      698
GO      537
PE      461
MA      403
ES      374
MT      147
CE      103
MS       59
RN       56
PB       40
RO       14
PI       12
SE       10
PA        8
AM        3
AC        1
Name: count, dtype: int64


### Export Cleaned Dataset For EDA

Exporting my cleaned dataset to a .csv for my next phase of Exploratory Data Analysis. 

In [50]:
print(df_1.to_csv('olist_dataset_clean.csv', index=False))
print("Document was successfully exported.")

None
Document was successfully exported.
